# Recipe Search Engine

A from-scratch exploration of classic information-retrieval techniques, applied to a database of ~50,000 recipes. The goal is to search for recipes by ingredient/keyword query and compare how different retrieval strategies trade off **speed** against **result quality**:

- Term-document incidence matrix (sklearn vs. a hand-rolled dense implementation)
- Inverted index built with a map-reduce style pipeline
- Inverted index backed by a binary search tree, implemented from scratch
- Jaccard similarity ranking
- TF-IDF ranking with cosine similarity
- A production-style search index using [Whoosh](https://whoosh.readthedocs.io/) with BM25F scoring

Every method is benchmarked on the same queries so the latency/quality numbers are directly comparable.

## Setup

In [3]:
#Install libraries and add imports
import sys, subprocess, pkg_resources

def install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package, "--quiet"])

required={'pandas', 'numpy', 'nltk', 'scikit-learn', 'whoosh'}
installed={pkg.key for pkg in pkg_resources.working_set}
for pkg in required - installed:
    install(pkg)

import pandas as pd
import numpy as np
import re, time, os, shutil
from collections import defaultdict

import nltk
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## Data Loading & Preprocessing

In [5]:
#Load the dataset
df=pd.read_csv('../data/food_recipes.csv')

#Pre-process the data
STOPWORDS=set(stopwords.words('english'))
lemmatiser=WordNetLemmatizer()
token_pattern=re.compile(r'[A-Za-z]{2,}')

def clean(text: str) -> str:
    tokens=token_pattern.findall(text.lower())
    tokens=[t for t in tokens if t not in STOPWORDS]
    tokens=[lemmatiser.lemmatize(t) for t in tokens]
    return ' '.join(tokens)

df=df.dropna(subset=['title', 'directions']).reset_index(drop=True)
df['clean']=df['directions'].apply(clean)

#Display last 5 rows
df[['title', 'directions', 'clean']].tail(5)

The above code pre-processes the data through the following steps:

1. Drop incomplete rows: Any recipe missing a title or directions is removed before processing. Incomplete records cannot be meaningfully indexed and would cause errors downstream.

2. Lowercase: All text is converted to lowercase so that "Butter," "BUTTER," and "butter" all collapse to the same token. This reduces vocabulary size and ensures case-insensitive matching.

3. Keep only alphabetic tokens: Keep only alphabetic tokens of 2+ characters and strip out punctuation, numbers, and special characters. These rarely carry semantic meaning in a bag-of-words model and can introduce noise.

4. Remove stopwords: Very common words like "the", "a", and "in" are filtered out. These appear in almost every recipe and do not help discriminate one document from another, so removing them focuses the index on more informative terms.

5. Lemmatisation: Each word is reduced to its canonical base form. For example, "cooking," "cooked," and "cooks" all become "cook." This ensures different inflections of the same word are counted together, improving both recall and consistency across queries.

## Term-Document Incidence Matrix

In [6]:
#Incidence matrix using sklearn CountVectorizer
bin_vectorizer=CountVectorizer(binary=True)
X_bin=bin_vectorizer.fit_transform(df['clean'])

#Display sklearn matrix (first 5 documents and first 10 terms)
vocab=bin_vectorizer.get_feature_names_out()
pd.DataFrame(X_bin[:5, :10].toarray(), columns=vocab[:10])

In [7]:
#Barebone incidence matrix
all_words=[]
for doc in df['clean']:
    for word in doc.split():
        all_words.append(word)
all_terms=sorted(set(all_words))

term_to_idx={t: i for i, t in enumerate(all_terms)}

bare_matrix=np.zeros((len(df), len(all_terms)), dtype=np.uint8)
for doc_id, text in enumerate(df['clean']):
    for term in set(text.split()):
        bare_matrix[doc_id, term_to_idx[term]]=1

#Display barebone matrix (first 5 documents and first 10 terms)
pd.DataFrame(bare_matrix[:5, :10], columns=all_terms[:10])

In [8]:
#Boolean search

#sklearn matrix version
def boolean_search_sklearn(query: str, vectorizer, matrix):
    terms=query.split()
    stack=[]
    vocab=vectorizer.vocabulary_
    for token in terms:
        if token in ('AND', 'OR', 'NOT'):
            stack.append(token)
        else:
            vec=np.zeros(matrix.shape[0], dtype=bool)
            if token.lower() in vocab:
                col=vocab[token.lower()]
                vec=matrix[:, col].toarray().ravel().astype(bool)
            stack.append(vec)
        while len(stack) >= 3 and isinstance(stack[-1], np.ndarray) and stack[-2] in ('AND', 'OR') and isinstance(stack[-3], np.ndarray):
            right, op, left=stack.pop(), stack.pop(), stack.pop()
            if op == 'AND':
                stack.append(left & right)
            else:
                stack.append(left | right)
        if len(stack) >= 2 and stack[-1] == 'NOT' and isinstance(stack[-2], np.ndarray):
            op=stack.pop()
            vec=stack.pop()
            stack.append(~vec)
    if stack and isinstance(stack[0], np.ndarray):
        return np.nonzero(stack[0])[0]
    return []

#barebone matrix version
def boolean_search_bare(query: str, term_to_idx, matrix):
    terms=query.split()
    stack=[]
    for token in terms:
        if token in ('AND', 'OR', 'NOT'):
            stack.append(token)
        else:
            vec=np.zeros(matrix.shape[0], dtype=bool)
            if token.lower() in term_to_idx:
                col=term_to_idx[token.lower()]
                vec=matrix[:, col].astype(bool)
            stack.append(vec)
        while len(stack) >= 3 and isinstance(stack[-1], np.ndarray) and stack[-2] in ('AND', 'OR') and isinstance(stack[-3], np.ndarray):
            right, op, left=stack.pop(), stack.pop(), stack.pop()
            if op == 'AND':
                stack.append(left & right)
            else:
                stack.append(left | right)
        if len(stack) >= 2 and stack[-1] == 'NOT' and isinstance(stack[-2], np.ndarray):
            op=stack.pop()
            vec=stack.pop()
            stack.append(~vec)
    if stack and isinstance(stack[0], np.ndarray):
        return np.nonzero(stack[0])[0]
    return []

QUERY="butter AND sugar"

t0=time.time()
res_sklearn=boolean_search_sklearn(QUERY, bin_vectorizer, X_bin)
t_sklearn=time.time() - t0

t0=time.time()
res_bare=boolean_search_bare(QUERY, term_to_idx, bare_matrix)
t_bare=time.time() - t0

print(f"sklearn results: {len(res_sklearn)} documents in {t_sklearn:.6f}s")
print(f"Barebone results: {len(res_bare)} documents in {t_bare:.6f}s")

sklearn results: 5318 documents in 0.014559s
Barebone results: 5318 documents in 0.002781s


In [9]:
df.loc[res_bare[:3], ['title', 'directions']]

In [10]:
df.loc[res_sklearn[:3], ['title', 'directions']]

#### Comparison and justification

Both implementations return the same 5318 documents for the query, so they are equally correct. The difference is in efficiency. The barebone dense NumPy matrix is faster (0.002781s vs 0.014559s) because looking up a column in a plain array is a direct operation, while the sklearn matrix is stored in a sparse format that needs to be converted with .toarray() on each lookup. However, the sparse sklearn matrix only stores the 1s, so it uses far less memory and scales much better as the vocabulary grows, whereas the dense matrix stores every 0 and 1 and would consume a large amount of memory on a bigger corpus. So the barebone version can be faster for a single lookup on this dataset, but sklearn is the more memory-efficient and scalable choice overall.

## Inverted Index (Map-Reduce Style)

In [11]:
def map_function(doc_id, text):
    return [(term, doc_id) for term in set(text.split())]

In [12]:
def reduce_function(term, doc_ids):
    return (term, sorted(doc_ids))

In [13]:
def create_inverted_index_map_reduce(dataframe):
    mapped=[]
    for doc_id, text in enumerate(dataframe['clean']):
        mapped.extend(map_function(doc_id, text))

    grouped=defaultdict(list)
    for term, doc_id in mapped:
        grouped[term].append(doc_id)

    inverted_index={}
    for term, doc_ids in grouped.items():
        _, postings=reduce_function(term, doc_ids)
        inverted_index[term]=postings

    return inverted_index

In [14]:
def inverted_index_search(recipe_data, inverted_index, search_terms):
    try:
        #Find all indices where all terms appear (boolean AND operation)
        sets_of_indices=[set(inverted_index[term]) for term in search_terms if term in inverted_index]
        if not sets_of_indices:
            return pd.DataFrame()  #Return an empty DataFrame if no indices found

        #Intersect all sets of indices to find common ones
        valid_indices=set.intersection(*sets_of_indices)

        #Convert the set of valid indices to a list
        valid_indices_list=list(valid_indices)

        #Select and return the relevant recipes using the valid indices
        return recipe_data.loc[valid_indices_list, ['title', 'directions']]
    except KeyError as e:
        #Handle the case where one or more search terms are not in the matrix
        print(f"Warning: {str(e).strip('[]')} not found in recipe terms.")
        return pd.DataFrame()  #Return an empty DataFrame if any term is not found

In [15]:
#Build the index and search
inverted_index=create_inverted_index_map_reduce(df)
SEARCH_TERMS=["butter", "sugar"]
results=inverted_index_search(df, inverted_index, SEARCH_TERMS)
print(f"Inverted index results for {SEARCH_TERMS}:")
print(results.head(5))

Inverted index results for ['butter', 'sugar']:
                       title  \
0        No-Bake Nut Cookies   
32770  Crustless Coconut Pie   
32773       English Teaballs   
6        Rhubarb Coffee Cake   
11             Buckeye Candy   

                                              directions  
0      In a heavy 2-quart saucepan, mix brown sugar, ...  
32770  Beat eggs., Add sugar, flour, milk, coconut, b...  
32773  Cream butter and confectioners sugar., Add dry...  
6      Cream sugar and butter., Add egg and beat well...  
11     Mix sugar, butter and peanut butter., Roll int...  


## Inverted Index Backed by a Binary Search Tree

In [16]:
#Inverted index using a binary search tree, built from scratch
class BSTNode:
    def __init__(self, key):
        self.key=key
        self.postings=[]
        self.left=None
        self.right=None

class BST:
    def __init__(self):
        self.root=None

    def insert(self, key, doc_id):
        self.root=self._insert(self.root, key, doc_id)

    def _insert(self, node, key, doc_id):
        if node is None:
            node=BSTNode(key)
            node.postings.append(doc_id)
            return node
        if key<node.key:
            node.left=self._insert(node.left, key, doc_id)
        elif key>node.key:
            node.right=self._insert(node.right, key, doc_id)
        elif doc_id not in node.postings:
            node.postings.append(doc_id)
        return node

    def search(self, key):
        node=self.root
        while node:
            if key==node.key:
                return node.postings
            if key<node.key:
                node=node.left
            else:
                node=node.right
        return []

bst_index=BST()
for doc_id, text in enumerate(df['clean']):
    for term in set(text.split()):
        bst_index.insert(term, doc_id)

#Search function
def bst_search(data, bst, search_terms):
    sets=[]
    for term in search_terms:
        sets.append(set(bst.search(term)))
    if not sets:
        return pd.DataFrame()
    valid=set.intersection(*sets)
    return data.loc[list(valid), ['title', 'directions']]

#Print results
results_bst=bst_search(df, bst_index, SEARCH_TERMS)
print(f"BST search results for {SEARCH_TERMS}:")
print(results_bst.head(5))

#Service latency
n=10
times=[]
for _ in range(n):
    t0=time.time()
    bst_search(df, bst_index, SEARCH_TERMS)
    times.append(time.time() - t0)
avg_latency_bst=sum(times)/n
print(f"\nBST Average Service Latency over {n} runs: {avg_latency_bst:.6f}s")

BST search results for ['butter', 'sugar']:
                       title  \
0        No-Bake Nut Cookies   
32770  Crustless Coconut Pie   
32773       English Teaballs   
6        Rhubarb Coffee Cake   
11             Buckeye Candy   

                                              directions  
0      In a heavy 2-quart saucepan, mix brown sugar, ...  
32770  Beat eggs., Add sugar, flour, milk, coconut, b...  
32773  Cream butter and confectioners sugar., Add dry...  
6      Cream sugar and butter., Add egg and beat well...  
11     Mix sugar, butter and peanut butter., Roll int...  

BST Average Service Latency over 10 runs: 0.006475s


## Jaccard Similarity Search

In [17]:
def jaccard_similarity(set1, set2):
    #Calculate the Jaccard similarity between two sets
    intersection=len(set1 & set2)
    union=len(set1 | set2)
    if union == 0:
        return 0.0
    return intersection/union

def jaccard_similarity_search(recipe_data, inverted_index, search_terms):
    try:
        sets_of_indices=[set(inverted_index[term]) for term in search_terms if term in inverted_index]
        if not sets_of_indices:
            return pd.DataFrame()
        valid_indices=set.intersection(*sets_of_indices)
        valid_indices_list=list(valid_indices)
        result=recipe_data.loc[valid_indices_list, ['title', 'directions']]
        result['jaccard_similarity']=result['directions'].apply(lambda x: jaccard_similarity(set(x.split()), set(search_terms)))

        return result
    except KeyError as e:
        print(f"Warning: {str(e).strip('[]')} not found in recipe terms.")
        return pd.DataFrame()

In [18]:
#Run the search
results_jaccard=jaccard_similarity_search(df, inverted_index, SEARCH_TERMS)
print("Jaccard similarity search results:")
print(results_jaccard.sort_values('jaccard_similarity', ascending=False).head(5))

#Service latency (average over multiple runs)
times=[]
for _ in range(n):
    t0=time.time()
    jaccard_similarity_search(df, inverted_index, SEARCH_TERMS)
    times.append(time.time() - t0)
avg_latency_jaccard=sum(times)/n
print(f"\nJaccard Average Search Latency over {n} runs: {avg_latency_jaccard:.6f}s")

Jaccard similarity search results:
                         title  \
49537  Country Carrot Frosting   
23438      Fluffy French Toast   
48814      Momma'S Coffee Cake   
191           Cranberry Scones   
28599        Chocolate Cookies   

                                              directions  jaccard_similarity  
49537  Cream well the butter and cream cheese., Add s...            0.153846  
23438  Beat eggs, milk, flour, sugar and salt., Dip i...            0.133333  
48814  Melt 1/2 of butter with brown sugar and pour i...            0.133333  
191    Combine softened butter or margarine, confecti...            0.125000  
28599  Boil cocoa, sugar and butter for 3 minutes., S...            0.117647  

Jaccard Average Search Latency over 10 runs: 0.072148s


## TF-IDF Search

In [19]:
#TF-IDF search
tfidf_vectorizer=TfidfVectorizer()
X_tfidf=tfidf_vectorizer.fit_transform(df['clean'])

def tfidf_search(query, top_k=5):
    q_vec=tfidf_vectorizer.transform([clean(query)])
    sims=cosine_similarity(q_vec, X_tfidf).ravel()
    top=np.argsort(sims)[-top_k:][::-1]
    return list(zip(top, sims[top]))

QUERY_STR=" ".join(SEARCH_TERMS)
ranked=tfidf_search(QUERY_STR)
print(f"TF-IDF results for '{QUERY_STR}':")
for doc_id, score in ranked:
    print(f"  Score {score:.4f} - {df.loc[doc_id, 'title']}")

#Service latency (average over multiple runs)
times=[]
for _ in range(n):
    t0=time.time()
    tfidf_search(QUERY_STR)
    times.append(time.time() - t0)
avg_latency_tfidf=sum(times)/n
print(f"\nTF-IDF Average Search Latency over {n} runs: {avg_latency_tfidf:.6f}s")

TF-IDF results for 'butter sugar':
  Score 0.5651 - Peanut Butter Cookies
  Score 0.5309 - Icebox Cake No. 1
  Score 0.5243 - Cobbler
  Score 0.5170 - Banana Bread
  Score 0.4810 - One Cup Cobbler

TF-IDF Average Search Latency over 10 runs: 0.030171s


## Full-Text Search with Whoosh (BM25F)

In [ ]:
from whoosh import index
from whoosh.fields import Schema, TEXT, ID
from whoosh.qparser import MultifieldParser
from whoosh.scoring import BM25F

df_whoosh=df.sample(frac=0.1, random_state=42).reset_index(drop=True) #using 10% of the data
#Schema: ID for the document id, TEXT for title and directions
schema=Schema(doc_id=ID(stored=True), title=TEXT(stored=True), directions=TEXT(stored=True))

#Create the index, add each recipe, and commit
whoosh_dir="whoosh_recipe_idx"
if os.path.exists(whoosh_dir):
    shutil.rmtree(whoosh_dir)
os.mkdir(whoosh_dir)
ix=index.create_in(whoosh_dir, schema)
writer=ix.writer()
for i, row in df_whoosh.iterrows():
    writer.add_document(doc_id=str(i), title=row['title'], directions=row['clean'])
writer.commit()

#Search function
def whoosh_search(query_str, top_k=5):
    with ix.searcher(weighting=BM25F()) as searcher:
        qp=MultifieldParser(["title", "directions"], ix.schema)
        q=qp.parse(query_str)
        results=searcher.search(q, limit=top_k)
        return [(r['doc_id'], r['title'], r['directions'], r.score) for r in results]

#Display title, directions and relevance score
print(f"Whoosh search results for '{QUERY_STR}':")
for doc_id, title, directions, score in whoosh_search(QUERY_STR):
    print(f"  Score {score:.4f} - {title}")
    print(f"    {directions[:100]}...")

Whoosh search results for 'butter sugar':
  Score 12.4408 - Sugar Cookies
    cream oil sugar butter powdered sugar add vanilla egg sift dry ingredient blend well chill dough sha...
  Score 12.3243 - Sugar Cookies
    cream sugar butter softened margarine add egg mix well add flour salt baking powder creamed sugar mi...
  Score 11.9832 - Sugar Cookies
    cream butter sugar add egg milk flavoring sift flour mix baking powder cup flour combine mixture add...
  Score 11.9593 - Sugar Cookies
    cream butter sugar blend oil egg vanilla add dry ingredient roll small ball put ungreased cookie she...
  Score 11.7884 - Sugar Pie
    cream butter sugar add well beaten egg flour beat well add milk vanilla pour inch unbaked shell bake...


The BM25F scoring algorithm was used. BM25F is an improvement over plain TF-IDF since TF-IDF only measures term frequency without accounting for document length, which means longer recipes are unfairly ranked higher. BM25F fixes this using document length normalization and term frequency saturation, giving more accurate relevance scores.

In [21]:
#Average indexing time over multiple runs
index_times=[]
for _ in range(n):
    t0=time.time()
    if os.path.exists(whoosh_dir):
        shutil.rmtree(whoosh_dir)
    os.mkdir(whoosh_dir)
    ix2=index.create_in(whoosh_dir, schema)
    w=ix2.writer()
    for i, row in df_whoosh.iterrows():
        w.add_document(doc_id=str(i), title=row['title'], directions=row['clean'])
    w.commit()
    index_times.append(time.time() - t0)
ix=index.open_dir(whoosh_dir)
avg_latency_index=sum(index_times)/n

#Average search time over multiple runs
search_times=[]
for _ in range(n):
    t0=time.time()
    whoosh_search(QUERY_STR)
    search_times.append(time.time() - t0)
avg_latency_whoosh=sum(search_times)/n

print(f"Avg indexing time ({n} runs): {avg_latency_index:.4f}s")
print(f"Avg search time ({n} runs): {avg_latency_whoosh:.6f}s")

#Space complexity: total size of the index directory on disk
total_size=0
for file in os.listdir(whoosh_dir):
    size=os.path.getsize(os.path.join(whoosh_dir, file))
    total_size+=size
    print(f"  {file}: {size} bytes")
print(f"\nTotal index size: {total_size/1024:.2f} KB")

Avg indexing time (10 runs): 3.8060s
Avg search time (10 runs): 0.016190s
  _MAIN_1.toc: 1889 bytes
  MAIN_7p4xumltb79zu5e2.seg: 4170953 bytes
  MAIN_WRITELOCK: 0 bytes

Total index size: 4075.04 KB


The total index directory is 4075.04 KB for 10% of the data. It is made up of a few files:

- .seg (segment) file holds the actual index content: the postings lists, term data, and stored fields. It is by far the largest file (about 4.17 MB here) and grows with the amount of text indexed.
- .toc (table of contents) file stores metadata about the index, such as the schema and which segments exist. It is very small (about 1.9 KB).
- WRITELOCK file is an empty lock file that stops two processes writing to the index at once, so it takes up no space (0 bytes).

## Performance Comparison

In [22]:
print("Average search latencies:")
print(f"Boolean (barebone):  {t_bare:.4f}s")
print(f"Boolean (sklearn):   {t_sklearn:.4f}s")
print(f"BST inverted index:  {avg_latency_bst:.4f}s")
print(f"Whoosh (BM25F):      {avg_latency_whoosh:.4f}s")
print(f"TF-IDF:              {avg_latency_tfidf:.4f}s")
print(f"Jaccard similarity:  {avg_latency_jaccard:.4f}s")

Average search latencies:
Boolean (barebone):  0.0028s
Boolean (sklearn):   0.0146s
BST inverted index:  0.0065s
Whoosh (BM25F):      0.0162s
TF-IDF:              0.0302s
Jaccard similarity:  0.0721s


Service Latency Comparison:

The matrix-based Boolean search and the tree-based inverted index are the fastest, because they only fetch and intersect postings with no scoring computation. Whoosh is in the middle, since reading from its prebuilt index file on disk adds some extra time compared to the purely in-memory methods. TF-IDF is slower because it multiplies the query vector against the whole document matrix. Jaccard similarity is the slowest because it computes a similarity score for every matching document on top of the postings lookup. So for per-query efficiency on this dataset, the inverted index and Boolean matrix methods are the most efficient.

Search Results Comparison:

The inverted index and BST return all recipes containing every query term but give no ranking, so the most relevant recipe cannot be identified. Jaccard ranks by how much the recipe overlaps with the query set, but treats all terms as equally important and favours short recipes. TF-IDF ranks better because it gives more weight to rarer and more informative terms. Whoosh with BM25F gave the most focused results (its top hits were all "Sugar Cookies" and "Sugar Pie", which strongly match the query), because it also handles term frequency saturation and document length normalization. Overall, TF-IDF and Whoosh/BM25F produced the most relevant rankings, while the Boolean matrix and inverted index methods are better for fast retrieval of all exact matches.

## Acknowledgements

Resources referenced while building this:

- [W3Schools: Binary Search Trees](https://www.w3schools.com/dsa/dsa_data_binarysearchtrees.php)
- [Whoosh documentation: searching](https://whoosh.readthedocs.io/en/latest/searching.html) and [scoring](https://whoosh.readthedocs.io/en/latest/api/scoring.html)
- [BM25 vs TF-IDF: a deep dive comparison](https://www.myscale.com/blog/bm25-vs-tf-idf-deep-dive-comparison/)